what is Collaborative Filtering Systems

Collaborative Filtering is a recommendation technique that suggests items to users based on the preferences, ratings, or behavior of similar users. Instead of relying on item features, it identifies patterns in user–item interactions to predict what a user may like by learning from the collective behavior of many users.

Uses of Collaborative Filtering in Industry

Collaborative filtering is widely used across industries to deliver personalized experiences. Streaming platforms like Netflix and Prime Video recommend movies based on similar users’ viewing histories. E-commerce platforms such as Amazon suggest products based on past purchases and user behavior. Music platforms like Spotify recommend songs and playlists, while social media platforms use collaborative filtering to suggest friends, pages, or content.

In [ ]:
import os
os.listdir('/content')


['.config', 'sample_data']

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [ ]:
movies = pd.read_csv("/movies_metadata.csv", low_memory=False)
ratings = pd.read_csv("/ratings.csv")



In [ ]:
movies = movies[['id', 'title']]
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies.dropna(inplace=True)
movies['id'] = movies['id'].astype(int)

ratings = ratings[['userId', 'movieId', 'rating']]

In [ ]:
data = pd.merge(ratings, movies, left_on='movieId', right_on='id')
data.head()

,userId,movieId,rating,id,title
0,1,110,1.0,110,Three Colors: Red
1,1,147,4.5,147,The 400 Blows
2,1,858,5.0,858,Sleepless in Seattle
3,1,1246,5.0,1246,Rocky Balboa
4,1,1968,4.0,1968,Fools Rush In


In [ ]:
user_movie_matrix = data.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

user_movie_matrix.fillna(0, inplace=True)
user_movie_matrix.head()

title,!Women Art Revolution,$5 a Day,'Gator Bait,'R Xmas,'Twas the Night Before Christmas,10 Items or Less,10 Things I Hate About You,"10,000 BC",11'09''01 - September 11,12 Angry Men,...,Zombie Holocaust,Zozo,Zvenigora,eXistenZ,xXx,¡A volar joven!,À nos amours,Ödipussi,Şaban Oğlu Şaban,Šíleně smutná princezna
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,4313,4314,4316,4317,4318,4319,4320,4321,4322,4323
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.000000,0.256100,0.136793,0.130637,0.0,0.221091,0.065518,0.187467,0.000000,...,0.011848,0.052238,0.000000,0.076261,0.0,0.000000,0.037395,0.020957,0.076004,0.082426
2,0.000000,1.000000,0.000000,0.062752,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.093754,0.043404,0.292844,0.023265,0.0,0.082174,0.053943,0.000000,0.067661,0.092430
3,0.256100,0.000000,1.000000,0.000000,0.161217,0.0,0.000000,0.043315,0.000000,0.000000,...,0.149867,0.103146,0.099282,0.052774,0.0,0.128152,0.144214,0.121232,0.070346,0.112674
4,0.136793,0.062752,0.000000,1.000000,0.079872,0.0,0.060527,0.061994,0.219348,0.000000,...,0.000000,0.015969,0.039350,0.077348,0.0,0.000000,0.177430,0.000000,0.081321,0.101041
5,0.130637,0.000000,0.161217,0.079872,1.000000,0.0,0.000000,0.000000,0.126993,0.093854,...,0.028640,0.189414,0.051859,0.151469,0.0,0.000000,0.031387,0.000000,0.086121,0.135217


In [ ]:
def get_similar_users(user_id, top_n=5):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    similar_users = similar_users.drop(user_id)
    return similar_users.head(top_n)


In [ ]:
def recommend_movies(user_id, num_recommendations=5):
    similar_users = get_similar_users(user_id)

    weighted_ratings = pd.Series(dtype=float)

    for sim_user, similarity in similar_users.items():
        user_ratings = user_movie_matrix.loc[sim_user]
        weighted_ratings = weighted_ratings.add(user_ratings * similarity, fill_value=0)

    recommended_movies = weighted_ratings.sort_values(ascending=False)

    watched_movies = user_movie_matrix.loc[user_id]
    watched_movies = watched_movies[watched_movies > 0].index

    recommended_movies = recommended_movies.drop(watched_movies, errors='ignore')

    return recommended_movies.head(num_recommendations)

In [ ]:
user_id = user_movie_matrix.index[0]
recommendations = recommend_movies(user_id)

recommendations

,0
title,
5 Card Stud,3.446302
In Time,2.141445
My Name Is Bruce,2.106826
Rise of the Zombies,2.104477
Terminator 3: Rise of the Machines,2.104477


In [ ]:
for movie, score in recommendations.items():
    print(f"Movie: {movie} | Score: {round(score, 2)}")

Movie: 5 Card Stud | Score: 3.45
Movie: In Time | Score: 2.14
Movie: My Name Is Bruce | Score: 2.11
Movie: Rise of the Zombies | Score: 2.1
Movie: Terminator 3: Rise of the Machines | Score: 2.1


This project successfully demonstrates the implementation of a collaborative filtering based movie recommender system using Python. By leveraging user rating behavior and cosine similarity, the system provides personalized movie recommendations. This approach is widely used in real-world platforms such as Netflix and Amazon for content personalization.